In [ ]:
dbutils.widgets.text("source_run_id", "", "Completed SOURCE run ID")
dbutils.widgets.text("pilot_batch_id", "", "Pilot proposal batch ID")
dbutils.widgets.text("pilot_proposal_ids", "", "One or two reviewed proposal IDs, comma-separated")

In [0]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from pyspark.sql import functions as F

# Databricks Git folders also put the repository root on the Python path.
repo_dirs = (Path.cwd(), Path.cwd().parent, *(Path(path) for path in sys.path if path))
for repo_dir in repo_dirs:
    src_dir = repo_dir if repo_dir.name == "src" else repo_dir / "src"
    if (src_dir / "synthetic_banking" / "rules.py").is_file():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break
else:
    raise RuntimeError("Open Notebook 03 from a Databricks Git folder containing this repository and src/.")

from synthetic_banking.rules import apply_balance_rules, load_rule_proposals
from synthetic_banking.validation import (
    GENERATED_SCHEMA,
    expected_generated_schema,
    null_padding_type,
    validate_generated_batch,
)

requested_source_run_id = dbutils.widgets.get("source_run_id").strip()
pilot_batch_id = dbutils.widgets.get("pilot_batch_id").strip()
pilot_proposal_ids = tuple(
    proposal_id.strip()
    for proposal_id in dbutils.widgets.get("pilot_proposal_ids").split(",")
    if proposal_id.strip()
)
if pilot_batch_id or pilot_proposal_ids:
    if (
        not requested_source_run_id or not pilot_batch_id
        or not 1 <= len(pilot_proposal_ids) <= 2
        or len(pilot_proposal_ids) != len(set(pilot_proposal_ids))
    ):
        raise ValueError("Pilot runs require a SOURCE run ID, one batch ID, and one or two distinct proposal IDs.")

# Pin one completed SOURCE run for every profile read.
source_run = spark.sql("""
    SELECT run_id
    FROM workspace.synthetic_profile.profile_runs
    WHERE profile_type = 'SOURCE'
      AND status = 'COMPLETED'
      AND completed_at IS NOT NULL
      AND run_id IS NOT NULL
      AND (:requested_source = '' OR run_id = :requested_source)
    ORDER BY completed_at DESC, started_at DESC, run_id DESC
    LIMIT 1
""", args={"requested_source": requested_source_run_id}).first()

if source_run is None:
    if requested_source_run_id:
        raise ValueError(f"Run {requested_source_run_id} is unavailable or is not a completed SOURCE profile.")
    raise RuntimeError(
        "No completed SOURCE profiling run found in workspace.synthetic_profile.profile_runs."
    )

source_run_id = source_run["run_id"]


def load_source_profile(table_name):
    return spark.table(f"workspace.synthetic_profile.{table_name}").filter(
        F.col("run_id") == source_run_id
    )


# Null rates / data quality
source_quality = load_source_profile("column_quality")
column_quality = source_quality.toPandas()

# Numeric distributions
numeric_stats = (
    load_source_profile("numeric_profile")
    .join(
        source_quality,
        on=["run_id", "logical_table_name", "column_name"],
        how="inner",
    )
    .select(
        "logical_table_name", "column_name",
        "row_count", "null_count", "null_pct", "distinct_count",
        "min_value", "p01", "p05", "p25", "p50", "mean_value",
        "p75", "p95", "p99", "max_value", "stddev_value",
    )
    .toPandas()
)

# Categorical distributions
category_stats = (
    load_source_profile("category_profile")
    .select(
        "logical_table_name", "column_name", "category_value", "record_count", "proportion"
    )
    .toPandas()
)

# Date statistics
date_stats = load_source_profile("date_profile").toPandas()

# Relationships / correlations
dependency_stats = (
    load_source_profile("relationship_profile")
    .select(
        F.col("entity_name").alias("relationship_name"),
        "variable_1", "variable_2", "metric_type", "metric_value",
    )
    .toPandas()
)

# Structural relationships
structural_stats = (
    load_source_profile("cardinality_profile")
    .select("relationship_name", "cardinality", "entity_count", "proportion")
    .toPandas()
)

# Conditional distributions
conditional_stats = load_source_profile("conditional_profile").toPandas()

print(f"Profiles loaded for SOURCE run {source_run_id}.")

print("Numeric:", len(numeric_stats))
print("Categorical:", len(category_stats))
print("Dates:", len(date_stats))
print("Quality:", len(column_quality))
print("Dependencies:", len(dependency_stats))
print("Structural:", len(structural_stats))
print("Conditional:", len(conditional_stats))

Set n count for creation

In [0]:
N_CUSTOMERS = 10_000

SEED = 42

rng = np.random.default_rng(SEED)

print(f"Generating {N_CUSTOMERS:,} customers")

create synthetic customers

In [0]:
person_ids = [
    f"P{i:08d}"
    for i in range(1, N_CUSTOMERS + 1)
]

person_df = pd.DataFrame({
    "PARTY_ID": person_ids
})

person_df.head()

In [0]:
def generate_categorical(
    table_name,
    column_name,
    n
):
    profile = category_stats[
        (category_stats["logical_table_name"] == table_name)
        &
        (category_stats["column_name"] == column_name)
    ].copy()

    # handle NULLs separately later
    profile = profile[
        profile["category_value"].notna()
    ]

    if profile.empty:
        return np.array([None] * n, dtype=object)

    values = profile["category_value"].to_numpy()

    probabilities = (
        profile["proportion"]
        .astype(float)
        .to_numpy()
    )

    # Re-normalize after removing NULL category
    probabilities = (
        probabilities /
        probabilities.sum()
    )

    return rng.choice(
        values,
        size=n,
        p=probabilities
    )

In [0]:
person_categorical_columns = (
    category_stats[
        category_stats["logical_table_name"] == "PERSON"
    ]["column_name"]
    .unique()
)

for column_name in person_categorical_columns:

    if column_name == "PARTY_ID":
        continue

    person_df[column_name] = generate_categorical(
        "PERSON",
        column_name,
        N_CUSTOMERS
    )

person_df.head()

Generate numeric tables

In [0]:
def generate_numeric(
    table_name,
    column_name,
    n
):
    profile = numeric_stats[
        (numeric_stats["logical_table_name"] == table_name)
        &
        (numeric_stats["column_name"] == column_name)
    ]

    if profile.empty:
        return np.full(n, np.nan)

    row = profile.iloc[0]

    percentile_points = np.array([
        0.00,
        0.01,
        0.05,
        0.25,
        0.50,
        0.75,
        0.95,
        0.99,
        1.00
    ])

    values = np.array([
        row["min_value"],
        row["p01"],
        row["p05"],
        row["p25"],
        row["p50"],
        row["p75"],
        row["p95"],
        row["p99"],
        row["max_value"]
    ], dtype=float)

    # Remove missing percentile points
    valid = ~np.isnan(values)

    percentile_points = percentile_points[valid]
    values = values[valid]

    if len(values) == 0:
        return np.full(n, np.nan)

    if len(values) == 1:
        return np.full(n, values[0])

    random_percentiles = rng.random(n)

    return np.interp(
        random_percentiles,
        percentile_points,
        values
    )

In [0]:
person_numeric_columns = (
    numeric_stats[
        numeric_stats["logical_table_name"] == "PERSON"
    ]["column_name"]
    .unique()
)

for column_name in person_numeric_columns:

    if column_name in person_df.columns:
        continue

    person_df[column_name] = generate_numeric(
        "PERSON",
        column_name,
        N_CUSTOMERS
    )

Null probablity

In [0]:
def get_null_probability(
    table_name,
    column_name
):
    profile = column_quality[
        (column_quality["logical_table_name"] == table_name)
        &
        (column_quality["column_name"] == column_name)
    ]

    if profile.empty:
        return 0.0

    null_pct = profile.iloc[0]["null_pct"]

    if pd.isna(null_pct):
        return 0.0

    return float(null_pct) / 100

In [0]:
def apply_profiled_nulls(
    dataframe,
    table_name,
    protected_columns=None
):
    if protected_columns is None:
        protected_columns = []

    for column_name in dataframe.columns:

        if column_name in protected_columns:
            continue

        null_probability = get_null_probability(
            table_name,
            column_name
        )

        if null_probability <= 0:
            continue

        mask = (
            rng.random(len(dataframe))
            < null_probability
        )

        dataframe.loc[
            mask,
            column_name
        ] = None

    return dataframe

In [0]:
person_df = apply_profiled_nulls(
    person_df,
    "PERSON",
    protected_columns=["PARTY_ID"]
)

In [0]:
def generate_date(
    table_name,
    column_name,
    n
):
    profile = date_stats[
        (date_stats["logical_table_name"] == table_name)
        &
        (date_stats["column_name"] == column_name)
    ]

    if profile.empty:
        return pd.Series(
            [pd.NaT] * n
        )

    row = profile.iloc[0]

    earliest = pd.to_datetime(
        row["earliest_date"]
    )

    latest = pd.to_datetime(
        row["latest_date"]
    )

    if pd.isna(earliest) or pd.isna(latest):
        return pd.Series(
            [pd.NaT] * n
        )

    total_days = (
        latest - earliest
    ).days

    if total_days <= 0:
        return pd.Series(
            [earliest] * n
        )

    random_days = rng.integers(
        0,
        total_days + 1,
        size=n
    )

    return (
        earliest
        + pd.to_timedelta(
            random_days,
            unit="D"
        )
    )

In [0]:
person_date_columns = (
    date_stats[
        date_stats["logical_table_name"] == "PERSON"
    ]["column_name"]
    .unique()
)

for column_name in person_date_columns:

    person_df[column_name] = generate_date(
        "PERSON",
        column_name,
        N_CUSTOMERS
    )

In [0]:
if (
    "BIRTH_DT" in person_df.columns
    and
    "SINCE_DT" in person_df.columns
):

    invalid = (
        person_df["SINCE_DT"].notna()
        &
        person_df["BIRTH_DT"].notna()
        &
        (
            person_df["SINCE_DT"]
            <
            person_df["BIRTH_DT"]
        )
    )

    person_df.loc[
        invalid,
        "SINCE_DT"
    ] = person_df.loc[
        invalid,
        "BIRTH_DT"
    ]

Cardinality

In [0]:
def generate_cardinality(
    relationship_name,
    n,
    fallback=1
):
    profile = structural_stats[
        structural_stats["relationship_name"]
        == relationship_name
    ].copy()

    if profile.empty:
        return np.full(
            n,
            fallback,
            dtype=int
        )

    values = (
        profile["cardinality"]
        .astype(int)
        .to_numpy()
    )

    probabilities = (
        profile["proportion"]
        .astype(float)
        .to_numpy()
    )

    probabilities = (
        probabilities /
        probabilities.sum()
    )

    return rng.choice(
        values,
        size=n,
        p=probabilities
    )

Accounts per each customer

In [0]:
account_counts = generate_cardinality(
    "ACCOUNTS_PER_PARTY",
    N_CUSTOMERS,
    fallback=1
)

total_accounts = int(
    account_counts.sum()
)

print(
    f"{N_CUSTOMERS:,} customers"
)

print(
    f"{total_accounts:,} accounts"
)

In [0]:
account_ids = [
    f"A{i:09d}"
    for i in range(
        1,
        total_accounts + 1
    )
]

In [0]:
account_party_ids = np.repeat(
    person_df["PARTY_ID"].to_numpy(),
    account_counts
)

customer_account_rel_df = pd.DataFrame({
    "PARTY_ID": account_party_ids,
    "ACCT_ID": account_ids
})

In [0]:
account_df = pd.DataFrame({
    "ACCOUNT_ID": account_ids
})

Categorical col's

In [0]:
account_categorical_columns = (
    category_stats[
        category_stats["logical_table_name"]
        == "TKQ_ACCOUNT"
    ]["column_name"]
    .unique()
)

for column_name in account_categorical_columns:

    if column_name in ("ACCOUNT_ID", "BUSINESSEFFECTIVEDATE"):
        continue

    account_df[column_name] = (
        generate_categorical(
            "TKQ_ACCOUNT",
            column_name,
            total_accounts
        )
    )

Numeric col's

In [0]:
account_numeric_columns = (
    numeric_stats[
        numeric_stats["logical_table_name"]
        == "TKQ_ACCOUNT"
    ]["column_name"]
    .unique()
)

for column_name in account_numeric_columns:

    if column_name == "BUSINESSEFFECTIVEDATE" or column_name in account_df.columns:
        continue

    account_df[column_name] = (
        generate_numeric(
            "TKQ_ACCOUNT",
            column_name,
            total_accounts
        )
    )

In [0]:
account_balance_df = pd.DataFrame({
    "ACCOUNT_ID": account_ids
})

In [0]:
raw_balance = generate_numeric(
    "TKQ_ACCOUNT_BALANCE",
    "CURRENT_BALANCE_AMT",
    total_accounts
)

raw_balance = np.maximum(
    raw_balance,
    0
)

In [0]:
if "CREDIT_LIMIT_AMT" in account_df.columns:

    credit_limit = (
        pd.to_numeric(
            account_df["CREDIT_LIMIT_AMT"],
            errors="coerce"
        )
        .fillna(0)
        .to_numpy()
    )

    # Allow some over-limit accounts,
    # but prevent absurd combinations
    current_balance = np.minimum(
        raw_balance,
        credit_limit * 1.25
    )

else:

    credit_limit = np.zeros(
        total_accounts
    )

    current_balance = raw_balance

In [0]:
account_balance_df[
    "CURRENT_BALANCE_AMT"
] = current_balance

In [0]:
if not account_df["ACCOUNT_ID"].equals(account_balance_df["ACCOUNT_ID"]):
    raise ValueError("Account and balance rows must align by ACCOUNT_ID before applying balance rules.")

rule_proposals = load_rule_proposals(spark, source_run_id, batch_id=pilot_batch_id)
if pilot_batch_id:
    approved_ids = {proposal.proposal_id for proposal in rule_proposals if proposal.approved}
    if approved_ids != set(pilot_proposal_ids):
        raise ValueError(
            f"Pilot batch {pilot_batch_id} must contain exactly the selected reviewed approvals "
            f"for SOURCE {source_run_id}. Found: {sorted(approved_ids)}."
        )
approved_count = sum(proposal.approved for proposal in rule_proposals)
print(f"Considering {len(rule_proposals)} proposals ({approved_count} approved) for SOURCE run {source_run_id}.")

balance_amounts, rule_events = apply_balance_rules(
    source_run_id, rule_proposals, credit_limit, current_balance,
    has_credit_limit="CREDIT_LIMIT_AMT" in account_df.columns,
)
for target_column, values in balance_amounts.items():
    account_balance_df[target_column] = values

for event in rule_events:
    target = f"{event.target_table}.{event.target_column}"
    if event.proposal_id is not None:
        target += f" (proposal {event.proposal_id}, batch {event.batch_id})"
    print(f"{event.outcome} {target}: {event.reason or event.implementation_name}")

if pilot_batch_id:
    applied_ids = {event.proposal_id for event in rule_events if event.applied}
    if applied_ids != set(pilot_proposal_ids):
        missed = sorted(set(pilot_proposal_ids) - applied_ids)
        raise RuntimeError(f"Pilot generation stopped before writes. Selected proposals did not apply: {missed}.")

In [0]:
account_df[
    "_UTILIZATION_RATE"
] = np.where(
    credit_limit > 0,
    current_balance / credit_limit,
    0
)

In [0]:
utilization_profile = conditional_stats[
    (
        conditional_stats["entity_name"]
        == "ACCOUNT"
    )
    &
    (
        conditional_stats["condition_variable"]
        == "UTILIZATION_RATE"
    )
    &
    (
        conditional_stats["outcome_variable"]
        == "PAST_DUE_INDICATO_CD"
    )
].copy()

In [0]:
delinquency_rates = {}

for _, row in utilization_profile.iterrows():

    delinquency_rates[
        row["condition_bucket"]
    ] = float(
        row["outcome_rate"]
    )

delinquency_rates

In [0]:
def get_delinquency_probability(
    utilization
):

    if utilization < 0.30:
        bucket = "0-30%"

    elif utilization < 0.60:
        bucket = "30-60%"

    elif utilization < 0.80:
        bucket = "60-80%"

    elif utilization <= 1.00:
        bucket = "80-100%"

    else:
        bucket = "100%+"

    # Use SQL rate if available
    if bucket in delinquency_rates:
        return delinquency_rates[bucket]

    # Temporary fallback
    return 0.05

In [0]:
delinquency_probability = (
    account_df["_UTILIZATION_RATE"]
    .apply(
        get_delinquency_probability
    )
    .to_numpy()
)

In [0]:
is_delinquent = (
    rng.random(total_accounts)
    <
    delinquency_probability
)

account_df[
    "PAST_DUE_INDICATO_CD"
] = np.where(
    is_delinquent,
    "Y",
    "N"
)

In [0]:
generated_dlq_count = generate_numeric(
    "TKQ_ACCOUNT",
    "DELINQUENCY_CNT",
    total_accounts
)

generated_dlq_count = np.nan_to_num(
    generated_dlq_count,
    nan=0
)

generated_dlq_count = np.maximum(
    np.round(
        generated_dlq_count
    ),
    1
)

account_df[
    "DELINQUENCY_CNT"
] = np.where(
    is_delinquent,
    generated_dlq_count,
    0
).astype(int)

In [0]:
def fill_profiled_fields(
    dataframe,
    table_name,
    protected_columns=None
):

    if protected_columns is None:
        protected_columns = []

    n = len(dataframe)

    # -----------------------
    # Categorical fields
    # -----------------------

    cat_columns = (
        category_stats[
            category_stats["logical_table_name"]
            == table_name
        ]["column_name"]
        .unique()
    )

    for column_name in cat_columns:

        if (
            column_name in protected_columns
            or
            column_name in dataframe.columns
        ):
            continue

        dataframe[column_name] = (
            generate_categorical(
                table_name,
                column_name,
                n
            )
        )

    # -----------------------
    # Numeric fields
    # -----------------------

    num_columns = (
        numeric_stats[
            numeric_stats["logical_table_name"]
            == table_name
        ]["column_name"]
        .unique()
    )

    for column_name in num_columns:

        if (
            column_name in protected_columns
            or
            column_name in dataframe.columns
        ):
            continue

        dataframe[column_name] = (
            generate_numeric(
                table_name,
                column_name,
                n
            )
        )

    # -----------------------
    # Date fields
    # -----------------------

    dt_columns = (
        date_stats[
            date_stats["logical_table_name"]
            == table_name
        ]["column_name"]
        .unique()
    )

    for column_name in dt_columns:

        if (
            column_name in protected_columns
            or
            column_name in dataframe.columns
        ):
            continue

        dataframe[column_name] = (
            generate_date(
                table_name,
                column_name,
                n
            )
        )

    dataframe = apply_profiled_nulls(
        dataframe,
        table_name,
        protected_columns
    )

    return dataframe

In [0]:
customer_account_rel_df = (
    fill_profiled_fields(
        customer_account_rel_df,
        "CUSTOMER_ACCOUNT_REL",
        protected_columns=[
            "PARTY_ID",
            "ACCT_ID"
        ]
    )
)

In [0]:
snapshot_date_column = "BUSINESSEFFECTIVEDATE"

account_balance_df = (
    fill_profiled_fields(
        account_balance_df,
        "TKQ_ACCOUNT_BALANCE",
        protected_columns=[
            "ACCOUNT_ID",
            snapshot_date_column,
            "CURRENT_BALANCE_AMT",
            "AVAILABLE_TO_BUY_AMT",
            "OVERLIMIT_AMT"
        ]
    )
)

# Sample one non-null effective date per account and reuse it on its balance row.
snapshot_dates = generate_date("TKQ_ACCOUNT_BALANCE", snapshot_date_column, len(account_df))
if pd.isna(snapshot_dates).any():
    snapshot_dates = generate_date("TKQ_ACCOUNT", snapshot_date_column, len(account_df))
if pd.isna(snapshot_dates).any():
    raise ValueError("The SOURCE profile needs BUSINESSEFFECTIVEDATE bounds for account snapshots.")

account_df[snapshot_date_column] = pd.DatetimeIndex(snapshot_dates)
dates_by_account_id = account_df.set_index("ACCOUNT_ID")[snapshot_date_column]
account_balance_df[snapshot_date_column] = account_balance_df["ACCOUNT_ID"].map(dates_by_account_id)

In [0]:
tradeline_counts = generate_cardinality(
    "TRADELINES_PER_CIF",
    N_CUSTOMERS,
    fallback=3
)

total_tradelines = int(
    tradeline_counts.sum()
)

In [0]:
tradeline_party_ids = np.repeat(
    person_df["PARTY_ID"].to_numpy(),
    tradeline_counts
)

In [0]:
tradeline_ids = [
    f"TL{i:010d}"
    for i in range(
        1,
        total_tradelines + 1
    )
]

In [0]:
tradelines_df = pd.DataFrame({
    "REQ-CIF-KEY":
        tradeline_party_ids,

    "REQ-CIF-TIE-BREAKER":
        1,

    "account_number":
        tradeline_ids
})

In [0]:
tradelines_df = fill_profiled_fields(
    tradelines_df,
    "TRADELINES",
    protected_columns=[
        "REQ-CIF-KEY",
        "REQ-CIF-TIE-BREAKER",
        "account_number"
    ]
)

In [0]:
customer_relationship_df = pd.DataFrame({
    "FROM_PARTY_ID": pd.Series(
        dtype="object"
    ),
    "TO_PARTY_ID": pd.Series(
        dtype="object"
    )
})

In [0]:
tables = {
    "PERSON":
        person_df,

    "CUSTOMER_ACCOUNT_REL":
        customer_account_rel_df,

    "CUSTOMER_RELATIONSHIP":
        customer_relationship_df,

    "TKQ_ACCOUNT":
        account_df,

    "TKQ_ACCOUNT_BALANCE":
        account_balance_df,

    "TRADELINES":
        tradelines_df
}

In [0]:
for table_name, df in tables.items():

    print(
        f"{table_name}: "
        f"{len(df):,} rows"
    )

In [0]:
expected_schema = expected_generated_schema(column_quality)

def add_missing_columns(df, logical_table_name):
    existing = {column.upper() for column in df.columns}
    for column in expected_schema[logical_table_name]:
        if column.upper() not in existing:
            df[column] = None
    return df

In [0]:
person_df = add_missing_columns(
    person_df,
    "PERSON"
)

account_df = add_missing_columns(
    account_df,
    "TKQ_ACCOUNT"
)

account_balance_df = add_missing_columns(
    account_balance_df,
    "TKQ_ACCOUNT_BALANCE"
)

customer_account_rel_df = add_missing_columns(
    customer_account_rel_df,
    "CUSTOMER_ACCOUNT_REL"
)

tradelines_df = add_missing_columns(
    tradelines_df,
    "TRADELINES"
)

In [0]:
from dataclasses import asdict
from uuid import uuid4

generated_schema = GENERATED_SCHEMA
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {generated_schema}")
generation_id = str(uuid4())


def publish_generation_state(status, validated_versions=None):
    spark.createDataFrame(
        [(generation_id, source_run_id, status, validated_versions, pilot_batch_id or None)],
        """generation_id STRING, source_run_id STRING, status STRING,
           validated_versions MAP<STRING, STRING>, proposal_batch_id STRING""",
    ).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"{generated_schema}.generation_state"
    )


publish_generation_state("RUNNING")


source_objects = {
    row["logical_table_name"]: row["source_object"]
    for row in spark.table("workspace.synthetic_profile.source_config").collect()
}


def save_generated_table(df, table_name):
    # Add typed NULLs after inference so ungenerated SOURCE columns survive Delta writes.
    null_columns = df.columns[df.isna().all()].tolist()
    populated = df.drop(columns=null_columns)
    if df.empty:
        output = spark.range(0)
    elif populated.shape[1] == 0:
        raise ValueError(f"{table_name} contains rows with no generated values.")
    else:
        output = spark.createDataFrame(populated)

    if null_columns:
        source_fields = {
            field.name.upper(): field.dataType
            for field in spark.table(f"workspace.{source_objects[table_name]}").schema.fields
        }
        expected_types = {
            column.upper(): data_type
            for column, data_type in expected_schema[table_name].items()
        }
        for column in null_columns:
            expected_type = expected_types.get(column.upper())
            if expected_type is None:
                raise ValueError(f"{table_name}.{column} has no expected type for an all-null column.")
            data_type = null_padding_type(expected_type, source_fields.get(column.upper()))
            output = output.withColumn(column, F.lit(None).cast(data_type))

    (
        output.select(*(F.col("`" + column.replace("`", "``") + "`") for column in df.columns))
        .write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{generated_schema}.{table_name}")
    )


save_generated_table(
    person_df,
    "PERSON"
)

save_generated_table(
    customer_account_rel_df,
    "CUSTOMER_ACCOUNT_REL"
)

save_generated_table(
    account_df,
    "TKQ_ACCOUNT"
)

save_generated_table(
    account_balance_df,
    "TKQ_ACCOUNT_BALANCE"
)

save_generated_table(
    tradelines_df,
    "TRADELINES"
)

# Publish the audit with the same generation ID before marking the batch completed.
rule_log = spark.createDataFrame(
    [
        {
            **asdict(event),
            "generation_id": generation_id,
            "source_run_id": source_run_id,
            "seed": SEED,
            # Spark Connect requires a list for an ARRAY field in a local row.
            "parent_columns": list(event.parent_columns),
        }
        for event in rule_events
    ],
    """generation_id STRING, source_run_id STRING, seed BIGINT,
       proposal_id STRING, batch_id STRING, target_table STRING, target_column STRING,
       rule_type STRING, outcome STRING, reason STRING, implementation_name STRING,
       execution_order INT, rows_processed BIGINT, parent_columns ARRAY<STRING>,
       approval_status STRING, proposed BOOLEAN, approved BOOLEAN, implemented BOOLEAN, applied BOOLEAN""",
).withColumn("logged_at", F.current_timestamp())
rule_log.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(
    f"{generated_schema}.rule_execution_log"
)

expected_counts = {
    "PERSON": N_CUSTOMERS,
    "CUSTOMER_ACCOUNT_REL": total_accounts,
    "TKQ_ACCOUNT": total_accounts,
    "TKQ_ACCOUNT_BALANCE": total_accounts,
    "TRADELINES": total_tradelines,
}
validated_versions = validate_generated_batch(
    spark, expected_schema, expected_counts, generation_id, source_run_id, len(rule_events),
)
publish_generation_state("COMPLETED", validated_versions)
print(f"Completed generation {generation_id}. Rule audit: {generated_schema}.rule_execution_log")

In [0]:
import os
import zipfile
import shutil

# Existing CSVs in the Volume
OUTPUT_DIR = "/Volumes/workspace/default/scotia/generated_row_level"

csv_paths = [
    f"{OUTPUT_DIR}/PERSON.csv",
    f"{OUTPUT_DIR}/CUSTOMER_ACCOUNT_REL.csv",
    f"{OUTPUT_DIR}/CUSTOMER_RELATIONSHIP.csv",
    f"{OUTPUT_DIR}/TKQ_ACCOUNT.csv",
    f"{OUTPUT_DIR}/TKQ_ACCOUNT_BALANCE.csv",
    f"{OUTPUT_DIR}/TRADELINES.csv"
]

# ---------------------------------------------------------
# Create ZIP locally first
# ---------------------------------------------------------

local_zip_path = "/tmp/RCAR_generated_row_level.zip"

with zipfile.ZipFile(
    local_zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    for csv_path in csv_paths:

        zip_file.write(
            csv_path,
            arcname=os.path.basename(csv_path)
        )

print("Local ZIP created:")
print(local_zip_path)

# ---------------------------------------------------------
# Copy completed ZIP into the Databricks Volume
# ---------------------------------------------------------

volume_zip_path = (
    "/Volumes/workspace/default/scotia/"
    "RCAR_generated_row_level.zip"
)

shutil.copyfile(
    local_zip_path,
    volume_zip_path
)

print("\nZIP copied to Volume:")
print(volume_zip_path)